# L5: Automate Event Planning

In this lesson, you will learn more about Tasks.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Crew, Task

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
from utils import load_env
load_env()

## crewAI Tools

In [4]:
from utils import BaiduWebSearchTool, ScrapeWebsiteTool
# Initialize the tools
search_tool = BaiduWebSearchTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="根据活动需求寻找并预订合适的场地",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "你对空间有敏锐感知，"
        "深谙活动后勤，擅长寻找并敲定"
        "符合活动主题、规模与预算限制的理想场地。"
    )
)

In [6]:
 # Agent 2: Logistics Manager
logistics_manager = Agent(
    role='Logistics Manager',
    goal="管理活动的全部后勤，包括餐饮与设备",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "你条理清晰、注重细节，"
        "确保活动从餐饮到设备搭建的"
        "每个后勤环节都完美执行，"
        "打造无缝的活动体验。"
    )
)

In [7]:
# Agent 3: Marketing and Communications Agent
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="有效推广活动并与参与者沟通",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "你富有创意、善于沟通，"
        "撰写引人入胜的宣传信息，"
        "与潜在参与者互动，"
        "最大化活动曝光与参与度。"
    )
)

## Creating Venue Pydantic Object

- Create a class `VenueDetails` using [Pydantic BaseModel](https://docs.pydantic.dev/latest/api/base_model/).
- Agents will populate this object with information about different venues by creating different instances of it.

In [8]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [9]:
venue_task = Task(
    description="在 {event_city} 寻找符合 {event_topic} 要求的场地。",
    expected_output="你所选定、用于承办活动的场地的全部详细信息。",
    # human_input=True,
    human_input=False,

    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

- `async_execution=True` 表示该任务可与**后续任务**并行执行。
- 新版 CrewAI 要求列表末尾**连续**的 async 任务不超过 1 个；若要让 `logistics_task` 与 `marketing_task` **同时**运行，需将二者都设为 async，并在最后追加一个**同步**的汇总任务（见下方 `consolidation_task`）。

In [10]:
logistics_task = Task(
    description="为 {tentative_date} 举行的、"
                 "预计 {expected_participants} 人参与的活动"
                 "协调餐饮与设备。",
    expected_output="确认所有后勤安排，"
                    "包括餐饮与设备搭建。",
    # human_input=True,
    human_input=False,

    async_execution=True,
    agent=logistics_manager
)

In [11]:
marketing_task = Task(
    description="推广 {event_topic}，"
                "目标吸引至少 {expected_participants} 名潜在参与者。",
    expected_output="以 Markdown 格式呈现的营销活动"
                    "与参与者互动报告。",
    async_execution=True,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [12]:
# 同步收尾任务：满足「末尾至多一个 async」的校验，并等待 logistics / marketing 并行完成后再汇总
consolidation_task = Task(
    description=(
        "场地、后勤与营销任务完成后，汇总 {event_topic} 在 {event_city} 的筹备状态，"
        "列出已完成事项与待跟进项。"
    ),
    expected_output="3-5 条要点的 Markdown 活动筹备状态摘要。",
    agent=venue_coordinator,
)

## Creating the Crew

**Note**: `logistics_task` 与 `marketing_task` 均设为 `async_execution=True`，在 `venue_task` 完成后会**并行**执行；末尾的 `consolidation_task` 为同步任务，用于等待二者结束并输出汇总（同时满足新版 CrewAI 的校验规则）。

In [13]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task,
           logistics_task,
           marketing_task,
           consolidation_task],
    
    verbose=True
)

# 末尾添加一个task，整合logistics_task和marketing_task

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

**Note 1**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Note 2**: 
- Since you set `human_input=True` for some tasks, the execution will ask for your input before it finishes running.
- When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [15]:
result = event_management_crew.kickoff(inputs=event_details)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 93bf6d88-b3ef-4f9d-98d6-0b3ad2cda336                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 在 San Francisco 寻找符合 Tech Innovation Conference 要求的场地。                                        │
│  ID: 866a987f-fe65-41ac-9469-a3f988b0690f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Task: 在 San Francisco 寻找符合 Tech Innovation Conference 要求的场地。                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  name='Tech Innovators Hub' address='123 Innovation Drive, San Francisco, CA 94105' capacity=500                │
│  booking_status='Available'                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 在 San Francisco 寻找符合 Tech Innovation Conference 要求的场地。                                        │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet                                                                                      │
│  Output: {"organic": [{"title": "EI Compendex Scopus ISI CNKI会议期刊论文集/从书征稿", "link":                  │
│  "https://www.bilibili.com/read/cv34146185/", "snippet": " 2024第6届全球经济、金融与人文研究国际会议(GEFHR      │
│  2024) 2024 6th International Conference on Global Economy, Finance and Humanities Research (GEFHR 2024)        │
│  2024年10月26-27日,纽约 DRP《Highlights in Business, Economics and Management》ISSN: 2957-952X出版              │
│  2024第3届商业管理、市场营销与对外贸易国际会议(BMMFT 2024) 2024 3rd International Conference on Business        │
│  Management, Marketing and Foreign Trade (BMMFT 2024) 2024年10月19-20日,法兰克福 WEP《Transactions on           │
│  Economics, Business and Management Research》ISSN: 2960-1789出版                                               │
│  2024第2届教育技术与心理科学研究国际会议(RETPS 2024) 20"}, {"title": "计算机视觉顶会 ECCV 2024                  │
│  揭榜:录用率或创新低,2395 篇论文中选", "link": "https://discovery.ithome.com/archiver/779/110.htm", "snippet":  │
│  " 计算机视觉顶会 ECCV 2024 揭榜:录用率或创新低,2395 篇论文中选 ECCV 2024 录用结果终于公布了! 一大早,ECC        │
│  组委会放出了所有被接受论文的 ID 名单,共录用了 2395 篇论文。 有网友估算了下,今年论文总提交量大约有 12600        │
│  篇,录用率是 18%。简直不敢相信今年 ECCV 的录用率如此之低,CVPR 2024 录用率还是 23.6%。 据统计,ECCV 2022 共有     │
│  5803 篇论文投稿,接收率为 28%。 再往前倒推,2020 年 ECCV 共收到有效投稿 5025 篇,接收论文 1361 篇,接收率为        │
│  27%。2018 年共有 2439 篇投稿,接收 776 篇,录用率为 31.8%。 ECCV                                                 │
│  表示,在接下来的几天里,还将公布最终的评审意见和元评审意见。还有论文 Poster / Oral 结果也将在随后揭晓。 今年,是  │
│  ECCV 召开的第 18 届顶会,将于 9 月 29 日-10 月 4 日在意大利米兰正式开幕。 ECCV(欧洲计算机视觉国际会议)创办于    │
│  1887 年,每两年举办一次。它与 CVPR(每年一届)ICCV(每两年一届)并"}, {"title":                                     │
│  "【IEEE官方列表会议】第十二届信息系统与计算技术国际会议(ISCTech 2024) ", "link":                               │
│  "https://m.sohu.com/sa/804654073_120643287", "snippet": " 会议简介 ISCTech 2024已进入IEEE官方会议列表:         │
│  https://conferences.ieee.org/conferences_events/conferences/conferencedetails/63666                            │
│  2024年第十二届信息系统与计算技术国际会议将于2024年11月8-11日在中国西安举行。该会议由长沙理工大学主办,同济大学  │
│  、西北工业大学协办,由IEEE西安分部提供技术支持。                                                                │
│  在即将举行的该次会议上,我们邀请到学术领域的知名教授将与参会者分享在信息系统与计算技术领域的最新创新和研究成果  │
│  。会议将主要以知名专家的主题演讲和作者的同行评审的论文报告为特色。此外,还将为参会者安排社交活动或学术访问,以   │
│  促进该领域研究人员之间的交流、讨论与合作。 ★被录用的文章将被收录在ISCTech                                      │
│  2024会议论文集里,由IEEE出版社出版,出版后所有文章都将进入IEEE Xplore数据库,并提交EI、Scopus检索。 ISCT 20222    │
│  ISBN-13: 979-8-3503-3293-3 ISCTech 2023 ISBN-13: 979-8-3503-4240-60"}, {"title":                               │
│  "活动邀请丨2024年IPBA仲裁日:科技创新与产业巨变下的仲裁 ", "link":                                              │
│  "https://mp.weixin.qq.com/s?__biz=MzA4MjU5MTUwOQ==&mid=2654084961&idx=2&sn=3397721cadbdfb768eabebaccc648b45&c  │
│  hksm=855869d7ab3d74e24584b9bd771386e842889a2fa9d3e4eb1f5144dca42811e21d8e9353428e&scene=27", "snippet":        │
│  "活动邀请丨2024年IPBA仲裁日:科技创新与产业巨变下的仲裁                                                         │
│  10月24-25日,2024年IPBA仲裁日活动将在深圳举行,活动由深圳国际仲裁院(SCIA)与环太平洋律师协会(IPBA)共同主办,竞天   │
│  公诚是大会赞助单位之一。本次大会以“科技创新与产业巨变下的仲裁”(Arbitration in the Face of Tech Innovation and  │
│  a Changing                                                                                                     │
│  Landscape)为主题,聚焦中国仲裁法修订、快速创新时代的仲裁、技术仲裁的常见问题、知识产权在科技仲裁中的应用、投资  │
│  合资和并购后的仲裁事项,以及投资仲裁在亚太的创新发展等热点、难点话题。目前已有来自美国、瑞士、日本、韩国、新加  │
│  坡等30多个国家和地区的近百名国际仲裁专家、仲裁员、资深律师及高校学者报名参会。诚挚邀请您参加本次大会,与国际法  │
│  律界共话知识产权与国际仲裁的新发展与新实践。 活动信息 <table><tr><th><section bdsfid=\"209\"><p                │
│  bdsfid=\"210\"><strong bdsfid=\"211\">时间</strong></p></section></t

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Tech Innovation Conference 筹备状态摘要                                                                      │
│                                                                                                                 │
│  ## 已完成事项                                                                                                  │
│  1.                                                                                                             │
│  **餐饮协调**：已与当地餐厅或餐饮供应商联系，确保为500人提供足够的食物和饮料，并根据活动时间表安排餐点。同时确  │
│  认了特殊饮食需求，并安排了餐饮运输和现场服务人员。                                                             │
│  2.                                                                                                             │
│  **设备搭建**：已与活动场地负责人沟通，了解可用的设备，并根据活动需求租赁额外设备。此外，已安排专业技术人员进   │
│  行设备安装和调试。                                                                                             │
│  3.                                                                                                             │
│  **营销活动与参与者互动**：通过社交媒体宣传、电子邮件营销、合作伙伴推广、在线广告以及线下活动联动等多种方式成   │
│  功吸引了620名注册参与者，并提升了会议的知名度。                                                                │
│                                                                                                                 │
│  ## 待跟进项                                                                                                    │
│  1. **交通和停车计划**：需进一步安排交通和停车计划，确保参与者能够顺利到达活动现场。                            │
│  2. **现场工作人员安排**：需要确定并安排现场工作人员，负责引导、接待和处理突发事件。                            │
│  3. **最后检查**：需进行最后的检查，确保所有后勤安排已落实到位。                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 场地、后勤与营销任务完成后，汇总 Tech Innovation Conference 在 San Francisco                             │
│  的筹备状态，列出已完成事项与待跟进项。                                                                         │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 93bf6d88-b3ef-4f9d-98d6-0b3ad2cda336                                                                       │
│  Final Output: # Tech Innovation Conference 筹备状态摘要                                                        │
│                                                                                                                 │
│  ## 已完成事项                                                                                                  │
│  1.                                                                                                             │
│  **餐饮协调**：已与当地餐厅或餐饮供应商联系，确保为500人提供足够的食物和饮料，并根据活动时间表安排餐点。同时确  │
│  认了特殊饮食需求，并安排了餐饮运输和现场服务人员。                                                             │
│  2.                                                                                                             │
│  **设备搭建**：已与活动场地负责人沟通，了解可用的设备，并根据活动需求租赁额外设备。此外，已安排专业技术人员进   │
│  行设备安装和调试。                                                                                             │
│  3.                                                                                                             │
│  **营销活动与参与者互动**：通过社交媒体宣传、电子邮件营销、合作伙伴推广、在线广告以及线下活动联动等多种方式成   │
│  功吸引了620名注册参与者，并提升了会议的知名度。                                                                │
│                                                                                                                 │
│  ## 待跟进项                                                                                                    │
│  1. **交通和停车计划**：需进一步安排交通和停车计划，确保参与者能够顺利到达活动现场。                            │
│  2. **现场工作人员安排**：需要确定并安排现场工作人员，负责引导、接待和处理突发事件。                            │
│  3. **最后检查**：需进行最后的检查，确保所有后勤安排已落实到位。                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

- Display the generated `venue_details.json` file.

In [16]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)

{'address': '123 Innovation Drive, San Francisco, CA 94105',
 'booking_status': 'Available',
 'capacity': 500,
 'name': 'Tech Innovators Hub'}


- Display the generated `marketing_report.md` file.

**Note**: After `kickoff` execution has successfully ran, wait an extra 45 seconds for the `marketing_report.md` file to be generated. If you try to run the code below before the file has been generated, your output would look like:

```
marketing_report.md
```

If you see this output, wait some more and than try again.

In [ ]:
from IPython.display import Markdown
Markdown("marketing_report.md")